### Data Cleaning and Quality Assessment
### Homework 1 – Tabular Data
### Hadi Seyed

### 2. Extra credit

In [12]:
import argparse
import os
import pandas as pd
import numpy as np
import sys


def parse_arguments():
    parser = argparse.ArgumentParser(
        description=(
            "Artificially add missing values to the Students.csv dataset."
        )
    )

    parser.add_argument(
        "input_csv",
        nargs="?", 
        default="Students.csv", 
        help=( 
            "Path to the input CSV file. " 
            "Default: Students.csv"
        )    
    )

    parser.add_argument(
        "-p",
        "--percent",
        type=float,
        default=15,
        help=(
            "Percentage of data instances to modify. "
            "Must be between 1 and 99. Default: 15."
        )
    )

    parser.add_argument(
        "-c",
        "--column",
        type=int,
        default=None,
        help=(
            "Column number in which missing values should "
            "be inserted. Uses 1-based numbering. "
            "If omitted, columns are selected randomly."
            "distributed across all columns."
        )
    )

    parser.add_argument(
        "-o",
        "--output",
        default=None,
        help=(
            "Path to output CSV. If omitted, '_na' is "
            "appended to the input filename."
        )
    )

    if "ipykernel" in sys.modules:  
        args = parser.parse_args([]) 
    else: 
        args = parser.parse_args()

    return args
    
    # args, unknown = parser.parse_known_args()
    # return args
    
    # return parser.parse_args()


def default_output_path(input_path):
    base, extension = os.path.splitext(input_path)

    if extension == "":
        extension = ".csv"

    return base + "_na" + extension


def add_missing_values(df, percent, column=None):
    
    # Number of rows to modify
    number_of_rows = len(df)

    number_to_modify = int(
        np.ceil(number_of_rows * percent / 100)
    )

    if number_to_modify == 0:
        number_to_modify = 1

    # # Make sure we do not try to select more rows than exist 
    # number_to_modify = min( 
    #     number_to_modify, 
    #     number_of_rows 
    # )
    
    # Randomly select rows without replacement
    rows = np.random.choice(
        number_of_rows,
        size=number_to_modify,
        replace=False
    )

    # A column was explicitly specified
    if column is not None:

        # Convert from 1-based column number to
        # Python's 0-based index.
        column_index = column - 1

        if (
            column_index < 0
            or column_index >= len(df.columns)
        ):
            raise ValueError(
                f"Column number must be between "
                f"1 and {len(df.columns)}."
            )

        # Insert missing values into selected rows
        df.iloc[rows, column_index] = np.nan

    # No column specified: randomly distribute
    # missing values among all columns.
    else:

        # Randomly select individual cells
        total_cells = number_to_modify * len(df.columns)

        cells = np.random.choice(
            len(df) * len(df.columns),
            size=total_cells,
            replace=False
        )

        for cell in cells:
            row = cell // len(df.columns)
            col = cell % len(df.columns)

            df.iloc[row, col] = np.nan

    return df


def main():
    args = parse_arguments()

    # Validate percentage
    if not 1 <= args.percent <= 99:
        raise ValueError(
            "Percentage must be between 1 and 99."
        )

    # Read input (Students.csv)
    df = pd.read_csv(args.input_csv)

    print("\nOriginal dataset:") 
    print(f"Rows: {len(df)}") 
    print(f"Columns: {len(df.columns)}") 
    
    print("\nColumns:") 
    for number, name in enumerate(df.columns, start=1): 
        print(f"{number}. {name}")
        
    # Add artificial missing values
    df = add_missing_values(
        df,
        args.percent,
        args.column
    )

    # Determine output path (filename)
    output_path = (
        args.output
        if args.output is not None
        else default_output_path(args.input_csv)
    )

    # Save modified dataset
    df.to_csv(
        output_path,
        index=False
    )

    # Display results
    print("\n--- Missing Value Generation Complete ---")
    
    print(
        f"Input file: {args.input_csv}"
    )

    print(
        f"Percentage requested: {args.percent}%"
    )

    if args.column is None:
        print(
            "Column: randomly distributed across columns"
        )
    else:
        print(
            f"Column: {args.column}"
            f"({df.columns[args.column - 1]})"
        )

    print(
        f"Output file: {output_path}"
    )

    print("\nMissing values by column:") 
    print(df.isna().sum())

if __name__ == "__main__":
    main()


Original dataset:
Rows: 10
Columns: 5

Columns:
1. Name
2. Age
3. GPA
4. Major
5. Credits

--- Missing Value Generation Complete ---
Input file: Students.csv
Percentage requested: 15%
Column: randomly distributed across columns
Output file: Students_na.csv

Missing values by column:
Name       2
Age        2
GPA        1
Major      4
Credits    1
dtype: int64
